# 02 · Preprocesamiento y construcción de ventanas

Este notebook aplica preprocesamiento ajustado solo con train, evita fuga de información y genera ventanas temporales multivariadas.

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

from scania_anomaly.config import load_config, ensure_directories
from scania_anomaly.utils.reproducibility import set_global_seed

config = load_config('../config/config.yaml')
ensure_directories(config)
set_global_seed(config['project']['seed'])

DRIVE_ROOT = Path(config['paths']['drive_root'])
RAW_DIR = Path(config['paths']['raw_dir'])
PROCESSED_DIR = Path(config['paths']['processed_dir'])
OUTPUTS_DIR = Path(config['paths']['outputs_dir'])
MODELS_DIR = Path(config['paths']['models_dir'])
METRICS_DIR = Path(config['paths']['metrics_dir'])
TABLES_DIR = Path(config['paths']['tables_dir'])
print(config['project']['name'], config['project']['version'])

from scania_anomaly.spark_session import create_spark_session
from scania_anomaly.data_loader import ScaniaDataLoader
from scania_anomaly.data_quality import DataQualityAnalyzer
from scania_anomaly.labels import attach_vehicle_labels, filter_normal_training_vehicles
from scania_anomaly.preprocessing import get_numeric_columns, apply_train_fitted_preprocessing
from scania_anomaly.windowing import TimeWindowBuilder
from scania_anomaly.experiment_tracking import save_json

spark = create_spark_session(config)
loader = ScaniaDataLoader.from_config(spark, config)


In [ ]:

train_op = loader.read_csv('train_operational')
val_op = loader.read_csv('validation_operational')
test_op = loader.read_csv('test_operational')
train_tte = loader.read_csv('train_tte')
val_labels = loader.read_csv('validation_labels')
test_labels = loader.read_csv('test_labels')

# Train se filtra a vehículos sin reparación en estudio para aprender patrones frecuentes/normalidad operacional.
train_normal = filter_normal_training_vehicles(train_op, train_tte, repair_col='in_study_repair')
validation_labeled = attach_vehicle_labels(val_op, val_labels, fill_unlabeled_with=-1)
test_labeled = attach_vehicle_labels(test_op, test_labels, fill_unlabeled_with=-1)


In [ ]:

exclude = config['preprocessing']['exclude_columns']
feature_cols = get_numeric_columns(train_normal, exclude=exclude)
print('N features iniciales:', len(feature_cols))

analyzer = DataQualityAnalyzer(train_normal)
missing_report = analyzer.missing_report()
constant_cols = analyzer.constant_columns(feature_cols) if config['preprocessing']['drop_constant_columns'] else []

train_prep, val_prep, test_prep, scaled_cols, metadata = apply_train_fitted_preprocessing(
    train_df=train_normal,
    validation_df=validation_labeled,
    test_df=test_labeled,
    feature_cols=feature_cols,
    missing_report=missing_report,
    constant_cols=constant_cols,
    max_missing_ratio=config['preprocessing']['max_missing_ratio'],
    vehicle_col=config['dataset']['vehicle_col'],
    time_col=config['dataset']['time_col'],
    label_col='y_true',
)

save_json(metadata.to_dict(), PROCESSED_DIR / config['preprocessing']['metadata_file'])
print('N features finales:', len(scaled_cols))
metadata.to_dict()


In [ ]:

mode = config['execution']['mode']
max_vehicles = config['execution']['max_vehicles_debug'] if mode == 'debug' else None

builder = TimeWindowBuilder(
    vehicle_col=config['dataset']['vehicle_col'],
    time_col=config['dataset']['time_col'],
    window_size=config['windowing']['window_size'],
    stride=config['windowing']['stride'],
    label_policy=config['windowing']['label_policy'],
)

train_windows = builder.build_from_spark(train_prep, scaled_cols, max_vehicles=max_vehicles, label_col=None)
validation_windows = builder.build_from_spark(val_prep, scaled_cols, max_vehicles=max_vehicles, label_col='y_true')
test_windows = builder.build_from_spark(test_prep, scaled_cols, max_vehicles=max_vehicles, label_col='y_true')

print('train:', train_windows.X.shape)
print('validation:', validation_windows.X.shape)
print('test:', test_windows.X.shape)


In [ ]:

TimeWindowBuilder.save_npz(PROCESSED_DIR / 'train_windows.npz', train_windows)
TimeWindowBuilder.save_npz(PROCESSED_DIR / 'validation_windows.npz', validation_windows)
TimeWindowBuilder.save_npz(PROCESSED_DIR / 'test_windows.npz', test_windows)
print('Ventanas guardadas en', PROCESSED_DIR)
